[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/matrixmultiplyconsortium/matrix-compute-colab/blob/main/01_quickstart_distributed_matmul.ipynb)

# 01 — Quickstart: Distributed Matrix Multiplication & Zero-Drain Trial Sandbox

Welcome to the interactive **Matrix Compute** quickstart notebook! In under 60 seconds, you will:

1. **Provision a Zero-Drain Trial Client (`mode="trial"`)**: Solve a lightweight client-side SHA-256 Hashcash puzzle (`mc_pow:...`) and mint a zero-value trial macaroon (`mc_trial_...`) scoped to our free seed workers — **no Lightning wallet, credit card, or API key required**.
2. **Execute a Distributed $4096 \times 4096$ FP32 Matrix Multiplication ($137.44\text{ GFLOPs}$)**: Partition matrices $A$ and $B$ into tiles, stream them over gRPC, verify worker Secp256k1 signatures, and assemble the result $C = A \times B$.
3. **Verify Integrity in $O(n^2)$ with Freivalds' Algorithm**: Benchmark probabilistic cryptographic verification ($A(Br) = Cr$) against $O(n^3)$ full recomputation and watch it catch Byzantine arithmetic tampering in milliseconds.
4. **Upgrade to Unbounded Mainnet Compute in 1 Line**: Pass `nwc_uri="nostr+walletconnect://..."` to unlock unconstrained dimensions across the decentralized GPU swarm.

In [1]:
# Step 1: Install matrix-compute-sdk (if running on a fresh Google Colab instance)
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("matrix_compute") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matrix-compute-sdk"])
    importlib.invalidate_caches()

import time
import numpy as np
import matrix_compute
from matrix_compute import (
    BandwidthProfiler,
    FreivaldsVerifier,
    MatrixComputeClient,
    MatrixDimensionError,
    PartitioningStrategy,
    distributed_matmul,
)

print(f"matrix-compute SDK v{matrix_compute.__version__} loaded successfully!")

matrix-compute SDK v0.1.0 loaded successfully!


## Step 2: Multiply Two $4096 \times 4096$ Matrices in `mode="trial"`

When initialized with `mode="trial"`, `MatrixComputeClient` enforces three zero-drain security guardrails:
- **Zero-Value Trial Macaroon (`mc_trial_...`)**: Locked satoshis are strictly `0 sats` — the settlement node never pays an outbound Lightning invoice for trial jobs.
- **Dimension Cap ($4096 \times 4096$)**: Prevents abuse while allowing realistic 137.4 GFLOP benchmarks.
- **Client-Side Hashcash PoW**: Protects the trial seed workers from automated spam.

In [2]:
# Generate two 4096 x 4096 FP32 matrices (67.1 MB each, 137.44 GFLOPs total GEMM workload)
rng = np.random.default_rng(42)
M = K = N = 4096
A = rng.standard_normal((M, K), dtype=np.float32)
B = rng.standard_normal((K, N), dtype=np.float32)

t0 = time.perf_counter()
with MatrixComputeClient(mode="trial") as client:
    C_dist = client.matmul(A, B, dtype="fp32")
    elapsed_s = time.perf_counter() - t0
    manifest = client.last_manifest
    trial_macaroon = client.last_trial_macaroon
    hashcash_stamp = client.last_hashcash_stamp
    pow_ms = client.last_hashcash_solve_ms

# Compare against local BLAS reference
t_ref0 = time.perf_counter()
C_ref = A @ B
ref_ms = (time.perf_counter() - t_ref0) * 1000.0
max_abs_err = float(np.max(np.abs(C_dist - C_ref)))

print("=== Zero-Drain Trial Execution Summary ===")
print(f"Matrix Dimensions     : ({M} x {K}) @ ({K} x {N}) [FP32]")
print(f"Total Workload        : {(2.0 * M * K * N) / 1e9:.2f} GFLOPs across {manifest.total_tasks} distributed tiles")
print(f"Hashcash PoW Stamp    : {hashcash_stamp} (solved in {pow_ms:.1f} ms)")
print(f"Trial Macaroon Prefix : {trial_macaroon[:48]}...")
print(f"Total Satoshis Spent  : {manifest.total_cost_sats} sats (Zero-Drain Sandbox)")
print(f"End-to-End Time       : {elapsed_s:.2f} s")
print(f"Max Absolute Error    : {max_abs_err:.3e} (allclose={np.allclose(C_dist, C_ref, rtol=1e-4, atol=1e-3)})")

=== Zero-Drain Trial Execution Summary ===
Matrix Dimensions     : (4096 x 4096) @ (4096 x 4096) [FP32]
Total Workload        : 137.44 GFLOPs across 8 distributed tiles
Hashcash PoW Stamp    : mc_pow:16:1793131563:025e6cafd5207467:1333 (solved in 2.1 ms)
Trial Macaroon Prefix : mc_trial_eyJjYXZlYXRzIjogeyJtb2RlIjogInRyaWFsIiw...
Total Satoshis Spent  : 0 sats (Zero-Drain Sandbox)
End-to-End Time       : 3.29 s
Max Absolute Error    : 1.373e-04 (allclose=True)


## Step 3: $O(n^2)$ Freivalds Verification vs. $O(n^3)$ Recomputation

Instead of trusting remote GPU workers blindly or re-running the $O(n^3)$ matrix multiplication locally, the SDK verifies every tile using **Freivalds' probabilistic algorithm** ($A(Br) = Cr$ over $k$ random Rademacher vectors $r \in \{-1, +1\}^n$) with soundness error $\le 2^{-k}$.

In [3]:
k_rounds = 10
t_ver0 = time.perf_counter()
is_valid, max_disc, dyn_bound, _ = FreivaldsVerifier.verify(
    A, B, C_dist, k_rounds=k_rounds, seed=2026
)
verify_ms = (time.perf_counter() - t_ver0) * 1000.0

# Profile network bandwidth for the 4096x4096 job
bw_profile = BandwidthProfiler.compare_strategies(
    m=M, k=K, n=N, tile_size=2048, num_strips=2, grid_rows=2, grid_cols=2, dtype="fp32"
)["output_grid_2d"]

print("=== Freivalds O(n^2) Verification & Bandwidth Profile ===")
print(f"Verification Verdict  : PASSED={is_valid} ({k_rounds} independent Rademacher/binary rounds)")
print(f"False-Accept Bound    : <= 2^-{k_rounds} ({2.0 ** (-k_rounds):.6%})")
print(f"Freivalds Verify Time : {verify_ms:.2f} ms vs. Local GEMM {ref_ms:.2f} ms ({ref_ms / max(verify_ms, 0.01):.1f}x faster)")
print(f"Max Observed Residual : {max_disc:.3e} (Higham dynamic bound: {dyn_bound:.3e})")
print(f"Bandwidth Transferred : {bw_profile.total_bytes_transferred / (1024**2):.1f} MiB (R_down = {bw_profile.download_redundancy_ratio:.2f})")

# Now inject Byzantine arithmetic corruption into a returned tile of C
C_corrupted = C_dist.copy()
C_corrupted[1024:1536, 1024:1536] += 15.0
corrupt_ok, corrupt_disc, corrupt_bound, _ = FreivaldsVerifier.verify(
    A, B, C_corrupted, k_rounds=k_rounds, seed=2026
)
print(f"\n[Byzantine Fault Injection] Corrupted tile C[1024:1536, 1024:1536] += 15.0:")
print(f"Verification Verdict  : PASSED={corrupt_ok} (Byzantine tampering immediately caught!)")
print(f"Discrepancy Detected  : {corrupt_disc:.2f} > Higham bound {corrupt_bound:.3e}")
assert is_valid is True and corrupt_ok is False

=== Freivalds O(n^2) Verification & Bandwidth Profile ===
Verification Verdict  : PASSED=True (10 independent Rademacher/binary rounds)
False-Accept Bound    : <= 2^-10 (0.097656%)
Freivalds Verify Time : 38.67 ms vs. Local GEMM 339.82 ms (8.8x faster)
Max Observed Residual : 8.789e-03 (Higham dynamic bound: 3.031e+03)
Bandwidth Transferred : 320.0 MiB (R_down = 0.00)

[Byzantine Fault Injection] Corrupted tile C[1024:1536, 1024:1536] += 15.0:
Verification Verdict  : PASSED=False (Byzantine tampering immediately caught!)
Discrepancy Detected  : 4095.01 > Higham bound 3.031e+03


## Step 4: Trial Sandbox Guardrails & 1-Line Mainnet Upgrade via NWC

The trial sandbox caps matrix dimensions at $4096 \times 4096$. When you are ready to scale to massive workloads ($16384 \times 16384+$ or multi-hour iterative spectral runs), simply switch to `mode="mainnet"` and pass a **Nostr Wallet Connect (`nwc_uri`)** connection string from any NWC-compatible Lightning wallet (Alby, Mutiny, Coinos, Cashu.me, Zeus).

In [4]:
# 1. Demonstrate deterministic 4096 cap enforcement in mode="trial"
A_oversized = np.zeros((4097, 64), dtype=np.float32)
B_oversized = np.zeros((64, 64), dtype=np.float32)

try:
    distributed_matmul(A_oversized, B_oversized, mode="trial")
except MatrixDimensionError as err:
    print("Expected Trial Cap Guardrail Triggered:")
    print(f"  -> {err}")

# 2. The 1-Line Mainnet Upgrade Path (Nostr Wallet Connect NIP-47)
MAINNET_UPGRADE_SNIPPET = '''
from matrix_compute import MatrixComputeClient

# Paste a Nostr Wallet Connect URI from Alby, Coinos, Cashu.me, or Mutiny:
with MatrixComputeClient(
    mode="mainnet",
    nwc_uri="nostr+walletconnect://<wallet_pubkey>?relay=wss://relay.damus.io&secret=<client_secret>",
) as mainnet_client:
    C_massive = mainnet_client.matmul(A_16k, B_16k, partitioning_strategy="OUTPUT_GRID_2D")
'''
print("\n=== 1-Line Mainnet Upgrade Path ===")
print(MAINNET_UPGRADE_SNIPPET.strip())

Expected Trial Cap Guardrail Triggered:
  -> PERMISSION_DENIED: Trial macaroon dimension exceeds 4096 cap (got A=(4097, 64), B=(64, 64)). Pass nwc_uri='nostr+walletconnect://...' in mode='mainnet' for unbounded dimensions. (shape_a=(4097, 64), shape_b=(64, 64))

=== 1-Line Mainnet Upgrade Path ===
from matrix_compute import MatrixComputeClient

# Paste a Nostr Wallet Connect URI from Alby, Coinos, Cashu.me, or Mutiny:
with MatrixComputeClient(
    mode="mainnet",
    nwc_uri="nostr+walletconnect://<wallet_pubkey>?relay=wss://relay.damus.io&secret=<client_secret>",
) as mainnet_client:
    C_massive = mainnet_client.matmul(A_16k, B_16k, partitioning_strategy="OUTPUT_GRID_2D")
